<a href="https://colab.research.google.com/github/rafiebazine/rag-image-captioning/blob/rafie/phase_01_data_exploration/download_and_extract_raw_coco_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Downloading and Extracting COCO 2017 Dataset
This notebook downloads the annotations, training dataset, and validation dataset from COCO 2017, and then extracts them.

# Download The data set + Unzip it.

In [1]:
# 1. Download annotations, train images, and validation images
!wget -q --show-progress http://images.cocodataset.org/annotations/annotations_trainval2017.zip
!wget -q --show-progress http://images.cocodataset.org/zips/train2017.zip
!wget -q --show-progress http://images.cocodataset.org/zips/val2017.zip

annotations_trainva 100%[===================>] 241.19M  22.1MB/s    in 13s     
train2017.zip       100%[===================>]  18.01G  19.9MB/s    in 15m 11s 
val2017.zip         100%[===================>] 777.80M  20.6MB/s    in 38s     


In [2]:
# 2. Extract all downloaded zip files
print("Extracting annotations...")
!unzip -q annotations_trainval2017.zip

Extracting annotations...


In [3]:
print("Extracting training dataset (this may take a while)...")
!unzip -q train2017.zip

Extracting training dataset (this may take a while)...


In [4]:
print("Extracting validation dataset...")
!unzip -q val2017.zip

Extracting validation dataset...


# Extract Sub-sets

## Extract Test subset
`100 images + 500 captions` + `1000 images + 5000 captions`

In [6]:
import json
import os
import random

# Fixed seed for exact reproducibility
random.seed(42)

# File paths for Google Colab environment
CAPTIONS_FILE = os.path.join("annotations", "captions_val2017.json")
INSTANCES_FILE = os.path.join("annotations", "instances_val2017.json")
IMAGES_DIR = "val2017"
OUTPUT_DIR = "splits"
OUTPUT_FILE_100 = os.path.join(OUTPUT_DIR, "test_100.json")
OUTPUT_FILE_1000 = os.path.join(OUTPUT_DIR, "test_1000.json")

# Category grouping rules
INDOOR_GROUPS = {"furniture", "kitchen", "indoor", "appliance", "electronic"}
OUTDOOR_GROUPS = {"outdoor", "vehicle", "animal", "sports"}


def read_json(file_path):
    with open(file_path, "r") as f:
        return json.load(f)


def map_category_contexts(instances_data):
    category_map = {}
    for cat in instances_data["categories"]:
        cat_id = cat["id"]
        group = cat["supercategory"].lower()

        if group in INDOOR_GROUPS:
            category_map[cat_id] = "indoor"
        elif group in OUTDOOR_GROUPS:
            category_map[cat_id] = "outdoor"
        else:
            category_map[cat_id] = "other"

    return category_map


def build_test_datasets():
    # Load annotations
    captions_data = read_json(CAPTIONS_FILE)
    instances_data = read_json(INSTANCES_FILE)
    category_map = map_category_contexts(instances_data)

    # 1. Step 1: Filter candidates with >= 5 captions that physically exist on disk
    captions_by_image = {}
    for ann in captions_data["annotations"]:
        captions_by_image.setdefault(ann["image_id"], []).append(ann["caption"])

    images_by_id = {img["id"]: img for img in captions_data["images"]}

    candidates = {}
    for img_id, captions in captions_by_image.items():
        if len(captions) >= 5 and img_id in images_by_id:
            file_name = images_by_id[img_id]["file_name"]
            image_path = os.path.join(IMAGES_DIR, file_name)

            if os.path.exists(image_path):
                candidates[img_id] = {
                    "image_id": img_id,
                    "file_name": file_name,
                    "height": images_by_id[img_id].get("height"),
                    "width": images_by_id[img_id].get("width"),
                    "captions": captions[:5],  # Truncate to exactly 5 captions
                }

    print(f"Step 1 Complete: {len(candidates)} valid candidate images found.")

    # 2. Extract object instances per candidate
    instances_by_image = {}
    for ann in instances_data["annotations"]:
        img_id = ann["image_id"]
        if img_id in candidates:
            instances_by_image.setdefault(img_id, []).append(ann["category_id"])

    # 3. Stratify candidates into 8 buckets
    buckets = {i: [] for i in range(1, 9)}

    for img_id, item in candidates.items():
        cat_ids = instances_by_image.get(img_id, [])
        obj_count = len(cat_ids)
        distinct_count = len(set(cat_ids))

        is_simple = 1 <= obj_count <= 2
        is_complex = obj_count >= 3
        if not (is_simple or is_complex):
            continue

        has_interaction = distinct_count >= 2

        contexts = [category_map.get(cid, "other") for cid in cat_ids]
        indoor_cnt = contexts.count("indoor")
        outdoor_cnt = contexts.count("outdoor")
        context = "indoor" if indoor_cnt > outdoor_cnt else "outdoor"

        if is_simple and not has_interaction and context == "indoor":
            bucket = 1
        elif is_simple and not has_interaction and context == "outdoor":
            bucket = 2
        elif is_simple and has_interaction and context == "indoor":
            bucket = 3
        elif is_simple and has_interaction and context == "outdoor":
            bucket = 4
        elif is_complex and not has_interaction and context == "indoor":
            bucket = 5
        elif is_complex and not has_interaction and context == "outdoor":
            bucket = 6
        elif is_complex and has_interaction and context == "indoor":
            bucket = 7
        elif is_complex and has_interaction and context == "outdoor":
            bucket = 8
        else:
            continue

        record = item.copy()
        record["stratified_category"] = bucket
        record["instance_count"] = obj_count
        record["distinct_categories"] = distinct_count
        record["context"] = context
        buckets[bucket].append(record)

    print("Candidate counts per bucket:")
    for b_id, items in buckets.items():
        print(f"  Bucket {b_id}: {len(items)} candidates")

    # Shuffle all buckets initially for randomized selection
    for b_id in buckets:
        random.shuffle(buckets[b_id])

    # 4. Sampling targets and adaptive redistribution
    targets_100 = {1: 13, 2: 13, 3: 13, 4: 13, 5: 12, 6: 12, 7: 12, 8: 12}
    targets_1000 = {1: 125, 2: 125, 3: 125, 4: 125, 5: 125, 6: 125, 7: 125, 8: 125}

    test_set_100 = []
    test_set_1000 = []

    # Allocate for test_100 first
    for bucket in range(1, 9):
        req = targets_100[bucket]
        available = buckets[bucket]
        selected = []
        for _ in range(min(req, len(available))):
            selected.append(available.pop())
        test_set_100.extend(selected)

    # If test_100 is still under 100 due to severe starvation, fill from remaining pools
    shortage_100 = 100 - len(test_set_100)
    if shortage_100 > 0:
        all_remaining = []
        for b_id in range(1, 9):
            all_remaining.extend(buckets[b_id])
        random.shuffle(all_remaining)
        for _ in range(min(shortage_100, len(all_remaining))):
            item = all_remaining.pop()
            test_set_100.append(item)
            # Remove from buckets pool
            buckets[item["stratified_category"]].remove(item)

    # Now sample for test_1000 with dynamic reallocation
    allotted_1000 = {b_id: 0 for b_id in range(1, 9)}
    shortage_1000 = 0

    # Try to fulfill targets per bucket
    for bucket in range(1, 9):
        req = targets_1000[bucket]
        available = buckets[bucket]
        taken = min(req, len(available))
        allotted_1000[bucket] = taken
        shortage_1000 += (req - taken)

    # Distribute the shortage of test_1000 dynamically to remaining categories with excess images
    while shortage_1000 > 0:
        eligible_buckets = [b for b in range(1, 9) if len(buckets[b]) > allotted_1000[b]]
        if not eligible_buckets:
            raise ValueError("Total candidate pool size across all buckets is too small to build test_1000.")
        # Choose bucket with largest unused candidate room to balance it
        best_bucket = max(eligible_buckets, key=lambda b: len(buckets[b]) - allotted_1000[b])
        allotted_1000[best_bucket] += 1
        shortage_1000 -= 1

    # Execute the actual sampling for test_1000 based on final allocations
    for bucket in range(1, 9):
        num_to_take = allotted_1000[bucket]
        for _ in range(num_to_take):
            test_set_1000.append(buckets[bucket].pop())

    # Sanity checks
    assert len(test_set_100) == 100, f"Expected 100 images, got {len(test_set_100)}"
    assert sum(len(x["captions"]) for x in test_set_100) == 500, "Expected 500 captions in test_100"

    assert len(test_set_1000) == 1000, f"Expected 1000 images, got {len(test_set_1000)}"
    assert sum(len(x["captions"]) for x in test_set_1000) == 5000, "Expected 5000 captions in test_1000"

    # Overlap and integrity validation
    ids_100 = {x["image_id"] for x in test_set_100}
    ids_1000 = {x["image_id"] for x in test_set_1000}
    assert ids_100.isdisjoint(ids_1000), "Overlap detected between test_100 and test_1000!"

    # Save splits
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    with open(OUTPUT_FILE_100, "w") as f:
        json.dump(test_set_100, f, indent=2)
    with open(OUTPUT_FILE_1000, "w") as f:
        json.dump(test_set_1000, f, indent=2)

    print(f"\n✅ Splits successfully created with dynamic fallback reallocation:")
    print(f"   - {OUTPUT_FILE_100} ({len(test_set_100)} images)")
    print(f"   - {OUTPUT_FILE_1000} ({len(test_set_1000)} images)")


if __name__ == "__main__":
    build_test_datasets()

Step 1 Complete: 5000 valid candidate images found.
Candidate counts per bucket:
  Bucket 1: 206 candidates
  Bucket 2: 590 candidates
  Bucket 3: 190 candidates
  Bucket 4: 532 candidates
  Bucket 5: 19 candidates
  Bucket 6: 210 candidates
  Bucket 7: 1483 candidates
  Bucket 8: 1722 candidates

✅ Splits successfully created with dynamic fallback reallocation:
   - splits/test_100.json (100 images)
   - splits/test_1000.json (1000 images)


## Extract RAG knowledge base & validation subset
`RAG Knowledge Base: 10k images + 50k captions`
`Validation subset: 100 images + 500 captions`


In [8]:
import json
import os

# File paths for Google Colab environment
TEST_100_FILE = os.path.join("splits", "test_100.json")
TEST_1000_FILE = os.path.join("splits", "test_1000.json")
CAPTIONS_FILE = os.path.join("annotations", "captions_train2017.json")
IMAGES_DIR = "train2017"
OUTPUT_DIR = "splits"
KB_OUTPUT_FILE = os.path.join(OUTPUT_DIR, "kb_10k.json")
VAL_OUTPUT_FILE = os.path.join(OUTPUT_DIR, "val_100.json")


def read_json(file_path):
    with open(file_path, "r") as f:
        return json.load(f)


def build_kb_and_val_datasets():
    # 1. Load reserved test image IDs from both test sets to prevent leakage
    test_100_data = read_json(TEST_100_FILE) if os.path.exists(TEST_100_FILE) else []
    test_1000_data = read_json(TEST_1000_FILE) if os.path.exists(TEST_1000_FILE) else []

    test_ids = {item["image_id"] for item in test_100_data} | {item["image_id"] for item in test_1000_data}
    print(f"Loaded {len(test_ids)} unique reserved test image IDs (from test_100 and test_1000) for exclusion.")

    # 2. Load train2017 captions
    captions_data = read_json(CAPTIONS_FILE)

    # Group captions by image_id
    captions_by_image = {}
    for ann in captions_data["annotations"]:
        captions_by_image.setdefault(ann["image_id"], []).append(ann["caption"])

    # Create images dictionary: {img_id : img}
    images_by_id = {img["id"]: img for img in captions_data["images"]}

    # Dictionary maps for unique image storage by image_id
    kb_10k = {}
    val_100 = {}

    # 3. Single-pass loop through candidate images
    for img_id, captions in captions_by_image.items():

        # Stop immediately when both targets are reached
        if len(kb_10k) == 10000 and len(val_100) == 100:
            break

        # Check 1: Exclude reserved test set images
        if img_id in test_ids:
            continue

        # Check 2: Exclude if fewer than 5 captions available Or
        # the image doesn't exist in the images dictionary
        if len(captions) < 5 or img_id not in images_by_id:
            continue

        # Check 3: Disk existence check
        file_name = images_by_id[img_id]["file_name"]
        image_path = os.path.join(IMAGES_DIR, file_name)

        # Skip when image doesn't exist physically on the disk
        if not os.path.exists(image_path):
            continue

        # Build candidate object with exactly 5 captions
        record = {
            "image_id": img_id,
            "file_name": file_name,
            "height": images_by_id[img_id].get("height"),
            "width": images_by_id[img_id].get("width"),
            "captions": captions[:5],  # Truncate to first 5 captions
        }

        # Phase A: Populate kb_10k dictionary up to 10,000 entries
        if len(kb_10k) < 10000:
            kb_10k[img_id] = record

        # Phase B: Populate val_100 dictionary up to 100 entries
        elif len(val_100) < 100:
            # Check 4: Confirm image is not in kb_10k or val_100
            if img_id not in kb_10k and img_id not in val_100:
                val_100[img_id] = record

    print("\nExtraction Summary:")
    print(f"  KB 10k size: {len(kb_10k)} images (50,000 captions)")
    print(f"  Val 100 size: {len(val_100)} images (500 captions)")

    # 4. Strict assertion audits

    # Guarantee the exact subset lengths
    assert len(kb_10k) == 10000, f"Expected 10,000 KB images, got {len(kb_10k)}"
    assert len(val_100) == 100, f"Expected 100 Val images, got {len(val_100)}"
    # Guarantee No overlap between sub-datasets
    assert set(kb_10k.keys()).isdisjoint(set(val_100.keys())), "Overlap detected between KB and Val!"
    assert set(kb_10k.keys()).isdisjoint(test_ids), "Test image leaked into KB set!"
    assert set(val_100.keys()).isdisjoint(test_ids), "Test image leaked into Val set!"

    # 5. Export JSON split files
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    with open(KB_OUTPUT_FILE, "w") as f:
        json.dump(list(kb_10k.values()), f, indent=2)

    with open(VAL_OUTPUT_FILE, "w") as f:
        json.dump(list(val_100.values()), f, indent=2)

    print("\n✅ Splits successfully created and saved:")
    print(f"   - {KB_OUTPUT_FILE}")
    print(f"   - {VAL_OUTPUT_FILE}")


if __name__ == "__main__":
    build_kb_and_val_datasets()

Loaded 1100 unique reserved test image IDs (from test_100 and test_1000) for exclusion.

Extraction Summary:
  KB 10k size: 10000 images (50,000 captions)
  Val 100 size: 100 images (500 captions)

✅ Splits successfully created and saved:
   - splits/kb_10k.json
   - splits/val_100.json
